# An Implementation of Modification of safPAKE  #

<h2> This Notebook gives an actual benchmark of our BIO-PAKE </h2>

In [3]:
import face_recognition
import matplotlib.pyplot as plt
import hashlib
import numpy as np
from tqdm import tqdm
import pandas as pd
import time
from bsp import CosineLSH
import python_bulletproofs

# 1. OPRF IMPLEMENTATION with Naor Reingold OPRF #


## 1.1 Lib Sodium Initialization and some EC-Operation Functions ##

In [4]:
import ctypes
import ctypes.util
import os
import hashlib
import secrets
from typing import List, Tuple

# --- 1. Libsodium Loading & Bindings ---

SCALAR_LEN = 32
POINT_LEN = 32

def load_sodium():
    for name in ("sodium", "libsodium"):
        path = ctypes.util.find_library(name)
        if path:
            try: return ctypes.CDLL(path)
            except: pass
    conda_prefix = os.environ.get("CONDA_PREFIX")
    if conda_prefix:
        candidates = [
            os.path.join(conda_prefix, "Library", "bin", "libsodium.dll"),
            os.path.join(conda_prefix, "lib", "libsodium.so"),
        ]
        for c in candidates:
            if os.path.exists(c): return ctypes.CDLL(c)
    raise OSError("libsodium not found")

sodium = load_sodium()
if hasattr(sodium, "sodium_init"):
    sodium.sodium_init()

# --- Bindings ---

# Scalar Math
sodium.crypto_core_ristretto255_scalar_random.argtypes = [ctypes.c_void_p]
try:
    sodium.crypto_core_ristretto255_scalar_mul.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_scalar_mul.restype = ctypes.c_int
except AttributeError:
    raise RuntimeError("libsodium version too old (missing scalar_mul).")
sodium.crypto_core_ristretto255_scalar_invert.argtypes = [ctypes.c_void_p, ctypes.c_void_p]

# Point Math
sodium.crypto_scalarmult_ristretto255_base.argtypes = [ctypes.c_void_p, ctypes.c_void_p] # P = n * G
sodium.crypto_scalarmult_ristretto255.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p] # P = n * Q

# NEW: Point Addition and Subtraction
try:
    # R = P + Q
    sodium.crypto_core_ristretto255_add.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_add.restype = ctypes.c_int
    
    # R = P - Q
    sodium.crypto_core_ristretto255_sub.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_sub.restype = ctypes.c_int
except AttributeError:
    raise RuntimeError("libsodium version too old (missing point add/sub).")


# --- Wrappers ---

def random_scalar() -> bytes:
    buf = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_random(ctypes.byref(buf))
    return bytes(buf)

def random_point() -> bytes:
    # To get a random valid point, we generate a random scalar and multiply by base
    return scalar_to_point(random_scalar())

def scalar_mul(x: bytes, y: bytes) -> bytes:
    z = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_mul(ctypes.byref(z), x, y)
    return bytes(z)

def scalar_invert(s: bytes) -> bytes:
    inv = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_invert(ctypes.byref(inv), s)
    return bytes(inv)

def scalar_to_point(s: bytes) -> bytes:
    p = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_scalarmult_ristretto255_base(ctypes.byref(p), s)
    return bytes(p)

def point_mul(scalar: bytes, point: bytes) -> bytes:
    out = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_scalarmult_ristretto255(ctypes.byref(out), scalar, point)
    return bytes(out)

def point_add(p: bytes, q: bytes) -> bytes:
    r = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_core_ristretto255_add(ctypes.byref(r), p, q)
    return bytes(r)

def point_sub(p: bytes, q: bytes) -> bytes:
    r = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_core_ristretto255_sub(ctypes.byref(r), p, q)
    return bytes(r)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))


def scalar_negate(s: bytes) -> bytes:
    """
    Computes the mathematical negation of a scalar modulo the curve order.
    Returns -s mod q.
    """
    neg = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_negate(ctypes.byref(neg), s)
    return bytes(neg)


"""
NOTE: this is super important:
    Since we are adding a paderson commitment now, we need to ensure 

"""

def _generate_global_H() -> bytes:
    seed = hashlib.sha512(b"safPAKE_Pedersen_Generator_H").digest()
    h_point = (ctypes.c_ubyte * 32)() # POINT_LEN is 32
    # sodium.crypto_core_ristretto255_from_hash is safe and deterministic
    sodium.crypto_core_ristretto255_from_hash(ctypes.byref(h_point), seed)
    return bytes(h_point)

GLOBAL_H = _generate_global_H()
SCALAR_ONE = b'\x01' + b'\x00' * 31

### 1.1.2 Benchmark on Each EC-operation ###

In [9]:
def run_microbenchmarks(iterations: int = 30) -> pd.DataFrame:
    """
    Measures the average execution time of libsodium EC wrapper functions.
    """
    print(f"=== Starting Microbenchmarks ({iterations} iterations per function) ===")
    
    # 1. Pre-generate valid inputs so we don't accidentally time the setup
    # (Assuming your wrappers are already loaded and working)
    s1 = random_scalar()
    s2 = random_scalar()
    p1 = random_point()
    p2 = random_point()
    
    # Ristretto255 operates on 32-byte chunks, so we create 32 random bytes for XOR
    b1 = os.urandom(32)
    b2 = os.urandom(32)

    # 2. Define the functions to test and their required arguments
    tests = [
        ("random_scalar", random_scalar, ()),
        ("Fixed-Based Multiplication", scalar_to_point, (s1,)),
        ("Variable-Based Multiplication", point_mul, (s1, p1)),
        ("EC Addition", point_add, (p1, p2)),
        ("EC Subtraction", point_sub, (p1, p2))
    ]

    results = []

    # 3. Run the benchmarks
    for func_name, func, args in tests:
        times = []
        for _ in range(iterations):
            start = time.perf_counter()
            func(*args)
            end = time.perf_counter()
            
            times.append(end - start)
            
        # Calculate the average time
        avg_time_sec = sum(times) / iterations
        
        results.append({
            "Function": func_name,
            "Avg Time (Seconds)": avg_time_sec,
            "Avg Time (Microseconds)": avg_time_sec * 1_000_000
        })

    # 4. Format into a pandas DataFrame and sort by slowest to fastest
    df = pd.DataFrame(results)
    df = df.sort_values(by="Avg Time (Microseconds)", ascending=False).reset_index(drop=True)
    
    return df


# Run the 30-iteration benchmark
benchmark_df = run_microbenchmarks(iterations=30)
# Print the beautifully formatted DataFrame
print("\nBenchmark Results (Sorted from slowest to fastest):")
print(benchmark_df.to_string(index=False))

=== Starting Microbenchmarks (30 iterations per function) ===

Benchmark Results (Sorted from slowest to fastest):
                     Function  Avg Time (Seconds)  Avg Time (Microseconds)
Variable-Based Multiplication            0.000133               132.763333
   Fixed-Based Multiplication            0.000053                52.880000
               EC Subtraction            0.000034                33.603333
                  EC Addition            0.000031                30.806667
                random_scalar            0.000005                 4.640000


## 1.2 BulletProof Interface ## 

### 1.2.1 Prover ###

In [ ]:
class ZKProver:
    """
    Client-side Prover: Holds the secret facial vector and generates Zero-Knowledge Proofs.
    """
    def __init__(self, k_bits: int):
        self.k_bits = k_bits
        # Calculate the maximum positive value for a signed k-bit integer
        # This is used to shift negative values into the strictly positive range required by Range Proofs.
        self.M = (1 << (k_bits - 1)) - 1

    def generate_vector_proofs(self, quantized_vector):
        """
        Generates Pedersen commitments and Bulletproofs for an entire quantized vector.
        """
        commitments = []
        proofs = []
        
        print(f"[Prover] Generating ZKP payload for {len(quantized_vector)} dimensions at {self.k_bits}-bit depth...")
        
        for i, val in enumerate(quantized_vector):
            # 1. Cast to standard Python int for Rust bridge safety
            original_val = int(val)
            
            # 2. Shift the value to be strictly positive (0 to 2^k - 1)
            shifted_val = original_val + self.M
            
            # 3. Safety Check
            if shifted_val < 0 or shifted_val >= (1 << self.k_bits):
                raise ValueError(f"[Prover] Fatal: Value {original_val} at index {i} is out of {self.k_bits}-bit bounds.")
            
            # 4. Generate the cryptographic commitment and proof
            comm, proof = python_bulletproofs.prove_range(shifted_val, self.k_bits)
            
            commitments.append(comm)
            proofs.append(proof)
            
        print("[Prover] Success! ZKP payload generated.")
        return commitments, proofs

### 1.2.2 Verifier ###

In [ ]:
class ZKVerifier:
    """
    Server-side Verifier: Receives cryptographic commitments and verifies their validity 
    without ever seeing the underlying facial data.
    """
    def __init__(self, k_bits: int):
        self.k_bits = k_bits

    def verify_vector_proofs(self, commitments, proofs) -> bool:
        """
        Verifies that all commitments represent values securely within the k-bit range.
        """
        if len(commitments) != len(proofs):
            print("[Verifier] REJECTED: Mismatched number of commitments and proofs.")
            return False
            
        print(f"[Verifier] Receiving {len(commitments)} proofs. Commencing verification...")
        
        for i in range(len(commitments)):
            comm = commitments[i]
            proof = proofs[i]
            
            # Call the Rust bridge to verify the math
            is_valid = python_bulletproofs.verify_range(comm, proof, self.k_bits)
            
            if not is_valid:
                print(f"[Verifier] REJECTED: Proof at index {i} failed cryptographic verification!")
                return False
                
        print("[Verifier] ACCEPTED: Entire vector cryptographically verified to be within bounds.")
        return True

## 1.3 El-Gamal Based Oblivious Transfer (OT) ##

In [10]:
class OTServer:
    def __init__(self):
        # The 'Setup' constant is now the globally defined generator H
        self.C = GLOBAL_H
        
        # Store for messages to be sent in the current batch
        self.message_pairs = []
        
        # NEW: Store the client's Pedersen commitments for ZKP verification
        self.client_commitments = []

    def load_messages(self, pairs: List[Tuple[bytes, bytes]]):
        self.message_pairs = pairs

    def handle_transfer(self, client_pk0_list: List[bytes]) -> List[Tuple[Tuple[bytes, bytes], Tuple[bytes, bytes]]]:
        if len(client_pk0_list) != len(self.message_pairs):
            raise ValueError("OT Mismatch: Client keys count != Message pairs count")

        # NEW: Save the Client's PK0s. These ARE the Pedersen Commitments.
        self.client_commitments = client_pk0_list.copy()

        response = []
        for i, (m0, m1) in enumerate(self.message_pairs):
            pk0 = client_pk0_list[i]
            
            # Derive PK1 = GLOBAL_H - PK0
            pk1 = point_sub(self.C, pk0)
            
            # --- Encryption remains exactly the same ---
            k0 = random_scalar()
            R0 = scalar_to_point(k0)
            S0 = point_mul(k0, pk0)
            key0 = hashlib.sha256(S0).digest()
            enc0 = xor_bytes(m0, key0) 
            
            k1 = random_scalar()
            R1 = scalar_to_point(k1)
            S1 = point_mul(k1, pk1)
            key1 = hashlib.sha256(S1).digest()
            enc1 = xor_bytes(m1, key1)
            
            response.append( ((R0, enc0), (R1, enc1)) )
            
        return response

class OTClient:
    def __init__(self):
        pass

    def prepare_keys(self, choice_bits: List[int]) -> Tuple[List[bytes], List[bytes], List[bytes]]:
        """
        Generates the OT keys based on choice bits.
        Returns: (List of PK0s to send, List of secrets x, List of blinding factors r for ZKP)
        """
        pk0_list = []
        secrets_x = []
        blinding_factors_r = [] # NEW: We must track r for the Zero-Knowledge Proof
        
        for b in choice_bits:
            x = random_scalar()
            secrets_x.append(x)
            
            # Calculate Key for Chosen Bit: PK_b = xG
            pk_chosen = scalar_to_point(x)
            
            # Calculate Keys to fit constraint: PK0 + PK1 = GLOBAL_H
            if b == 0:
                pk0 = pk_chosen
                r = x  # Blinding factor is x
            else:
                pk0 = point_sub(GLOBAL_H, pk_chosen)
                r = scalar_negate(x)  # Blinding factor is -x mod q
            
            pk0_list.append(pk0)
            blinding_factors_r.append(r)
            
        return pk0_list, secrets_x, blinding_factors_r

    def decrypt(self, response: List, secrets_x: List[bytes], choice_bits: List[int]) -> List[bytes]:
        # --- Decryption logic remains EXACTLY the same ---
        results = []
        for i, ((R0, enc0), (R1, enc1)) in enumerate(response):
            x = secrets_x[i]
            b = choice_bits[i]
            
            R, enc = (R1, enc1) if b == 1 else (R0, enc0)
            
            S = point_mul(x, R)
            key = hashlib.sha256(S).digest()
            m = xor_bytes(enc, key)
            
            results.append(m)
        return results

## 1.4 OPRF Implementation ##

In [11]:
class OPRF_Server:
    def __init__(self, input_bits: int):
        self.n = input_bits
        # NR Key
        self.a0 = random_scalar()
        self.a_vec = [random_scalar() for _ in range(self.n)]
        
        # Instantiate OT Server (Now uses GLOBAL_H internally)
        self.ot_server = OTServer()

    def process_request(self) -> bytes:
        """
        1. Generates blinding factors r_i
        2. Computes Correction
        3. Loads OT with (r_i, r_i * a_i)
        Returns: Correction_Factor (scalar)
        """
        r_vec = [random_scalar() for _ in range(self.n)]
        
        ot_messages = []
        for i in range(self.n):
            m0 = r_vec[i]
            m1 = scalar_mul(r_vec[i], self.a_vec[i])
            ot_messages.append((m0, m1))
        
        # Load into OT Server
        self.ot_server.load_messages(ot_messages)
        
        # Calculate Correction: a0 * prod(r)^-1
        prod_r = SCALAR_ONE
        for r in r_vec:
            prod_r = scalar_mul(prod_r, r)
        
        correction = scalar_mul(self.a0, scalar_invert(prod_r))
        return correction

    def get_client_commitments(self) -> List[bytes]:
        """
        NEW: Retrieves the Pedersen commitments submitted by the client during the OT phase.
        The server will pass these into the ZKP verifier.
        """
        return self.ot_server.client_commitments

class OPRF_Client:
    def __init__(self, input_bits: int):
        self.n = input_bits
        self.ot_client = OTClient()

    def evaluate(self, input_data: bytes, server_instance: OPRF_Server, correction: bytes):
        """
        Full OPRF evaluation flow.
        NEW Returns: (Final_OPRF_Point, List_of_Bits, List_of_Blinding_Factors)
        """
        # 1. Input -> Bits
        input_int = int.from_bytes(input_data, 'big')
        bits = [(input_int >> i) & 1 for i in range(self.n)]
        
        # 2. OT Request Generation
        # REMOVED: ot_C = server_instance.ot_server.C (We now use GLOBAL_H internally)
        
        # NEW: Unpack the 3 variables, including the blinding factors (r) for the ZKP
        pk0_list, secrets, blinding_factors_r = self.ot_client.prepare_keys(bits)
        
        # 3. OT Transfer (Network Simulation)
        # The Server secretly saves pk0_list as the commitments during this step!
        ciphertexts = server_instance.ot_server.handle_transfer(pk0_list)
        
        # 4. OT Decryption
        shares = self.ot_client.decrypt(ciphertexts, secrets, bits)
        
        # 5. OPRF Finalization
        # Result = Correction * Product(shares)
        P = SCALAR_ONE
        for s in shares:
            P = scalar_mul(P, s)
            
        final_scalar = scalar_mul(correction, P)
        
        # Map to point
        final_point = scalar_to_point(final_scalar)

        # NEW: Return the OPRF result AND the ZKP witness data
        return final_point, bits, blinding_factors_r

# Section X: OPRF computation analysis #

## Section X.1 Time Benchmark ##

In [12]:
import time
import os

def benchmark_oprf_time(N_bits: int, K_iterations: int) -> float:
    """
    Benchmarks the Naor-Reingold OPRF with ElGamal OT.
    
    Args:
        N_bits: The bit length of the OPRF input (e.g., 128).
        K_iterations: Number of times to run the protocol.
        
    Returns:
        Total time consumed (in seconds) for K iterations.
    """
    
    # 1. Setup Phase (Not included in timing, as this is one-time init)
    server = OPRF_Server(N_bits)
    client = OPRF_Client(N_bits)
    
    # Generate random inputs for the K runs to ensure realistic processing
    # Calculate byte length: ceil(N / 8)
    byte_len = (N_bits + 7) // 8
    inputs = [os.urandom(byte_len) for _ in range(K_iterations)]
    
    print(f"--- Starting Benchmark (N={N_bits}, K={K_iterations}) ---")
    
    # 2. Timing Phase
    start_time = time.time()
    
    for i in range(K_iterations):
        # A. Server prepares the OT batch and correction factor
        # (In a real protocol, this happens before or during the request)
        correction = server.process_request()
        
        # B. Client evaluates the OPRF (generates OT keys, retrieves, computes final)
        _ = client.evaluate(inputs[i], server, correction)
        
    end_time = time.time()
    
    total_time = end_time - start_time
    
    # print(f"Done. Total: {total_time:.4f}s")
    return total_time


def theoretical_oprf_time(N_bits: int, K_iterations: int, op_times: dict) -> float:
    """
    Calculates the theoretical execution time of K iterations of the OPRF
    based on individual operation microbenchmarks.
    
    Args:
        N_bits: Bit length of the OPRF input.
        K_iterations: Number of protocol runs.
        op_times: Dictionary containing the average time (in seconds) for each operation.
                  Keys expected: 'random_scalar', 'fixed_base', 'var_base', 'ec_sub'
                  
    Returns:
        Theoretical total time in seconds.
    """
    
    # 1. Count operations per single OPRF run (Server + Client combined)
    count_random = (4 * N_bits) + 1
    count_fixed  = (3 * N_bits) + 2
    count_var    = (3 * N_bits)
    count_sub    = (1.5 * N_bits) # Expected average (50% of bits are 1)
    
    # 2. Calculate time for one OPRF run
    time_per_run = (
        (count_random * op_times.get('random_scalar', 0)) +
        (count_fixed  * op_times.get('fixed_base', 0)) +
        (count_var    * op_times.get('var_base', 0)) +
        (count_sub    * op_times.get('ec_sub', 0))
    )
    
    # 3. Multiply by K iterations
    total_theoretical_time = time_per_run * K_iterations
    
    return total_theoretical_time

## Section X.2 Bandwidth benchmark ##

In [13]:
def calculate_bandwidth(N_bits: int, K_iterations: int) -> int:
    """
    Calculates the total bandwidth (in bits) required for K executions 
    of the Naor-Reingold OPRF with Bellare-Micali OT.

    Args:
        N_bits: The number of bits in the OPRF input (e.g., 128).
        K_iterations: The number of OPRF operations to perform.

    Returns:
        Total bandwidth in bits.
    """
    
    # --- Constants (libsodium / Ristretto255) ---
    SIZE_SCALAR = 256  # 32 bytes * 8
    SIZE_POINT  = 256  # 32 bytes * 8
    
    # --- Per-Execution Costs ---
    
    # 1. Server Setup (Sent once per execution)
    # Sends: Correction Factor (Scalar) + OT Setup C (Point)
    setup_bandwidth = SIZE_SCALAR + SIZE_POINT
    
    # 2. Client OT Request (Sent for every bit)
    # Sends: One Public Key (PK0) per bit
    client_request_per_bit = SIZE_POINT
    
    # 3. Server OT Response (Sent for every bit)
    # Sends: Two ciphertext pairs per bit.
    # Pair 0: (R0, Enc0) -> (Point, Scalar)
    # Pair 1: (R1, Enc1) -> (Point, Scalar)
    # Note: Enc is a masked scalar, so it is Scalar size.
    server_response_per_bit = 2 * (SIZE_POINT + SIZE_SCALAR)
    
    # --- Aggregation ---
    
    total_per_bit = client_request_per_bit + server_response_per_bit
    total_per_run = setup_bandwidth + (N_bits * total_per_bit)
    
    total_bandwidth_bits = K_iterations * total_per_run
    
    # --- Optional: Print breakdown for clarity ---
    # print(f"--- Bandwidth Calculation (N={N_bits}, K={K_iterations}) ---")
    # print(f"Per Bit Overhead:    {total_per_bit} bits")
    # print(f"Per Run Overhead:    {setup_bandwidth} bits (Setup)")
    # print(f"Total per OPRF Run:  {total_per_run} bits ({total_per_run/8/1024:.2f} KB)")
    # print(f"Grand Total:         {total_bandwidth_bits} bits ({total_bandwidth_bits/8/1024/1024:.2f} MB)")

    return total_bandwidth_bits

## Section X.3 Time and Bandwidth analysis for OPRF eval ##

In [14]:
# 1. Define the parameters based on your previous grid search
N_list = [64, 72, 80, 88, 96, 104]
K_list = [1, 3, 5, 7, 9, 11, 13]  # K represents the number of Bags


my_measured_times = {
    'random_scalar': 0.000002,  # e.g., 1 microsecond
    'fixed_base':    0.000052,  # e.g., 50 microseconds
    'var_base':      0.000148,  # e.g., 150 microseconds
    'ec_sub':        0.000039   # e.g., 2 microseconds
}

In [ ]:
results = []
    
total_runs = len(N_list) * len(K_list)
current_run = 1

print(f"=== Starting OPRF Grid Evaluation ({total_runs} total combinations) ===\n")

# 2. Iterate through every combination of N and K
for n_bits in N_list:
    for k_iters in K_list:
        print(f"[{current_run}/{total_runs}] Evaluating N={n_bits} bits, K={k_iters} iterations...")
        
        # --- A. Calculate Theoretical Bandwidth ---
        bw_bits = calculate_bandwidth(n_bits, k_iters)
        bw_kb = bw_bits / (8 * 1024) # Convert to Kilobytes for readability
        
        # --- B. Benchmark Cryptographic Execution Time ---
        t_seconds = benchmark_oprf_time(n_bits, k_iters)

        expected_time = theoretical_oprf_time(n_bits, k_iters, my_measured_times)

        
        # Store the results
        results.append({
            'N_Bits': n_bits,
            'K_Iterations (Bags)': k_iters,
            'Bandwidth_Bits': bw_bits,
            'Bandwidth_KB': bw_kb,
            'Time_Seconds': t_seconds,
            'theory_time(seconds)': expected_time
        })
        
        current_run += 1

# 3. Convert results to a pandas DataFrame for nice formatting
df_results = pd.DataFrame(results)

# 4. Save to CSV
output_filename = "oprf_grid_performance.csv"
df_results.to_csv(output_filename, index=False)

print(f"\n=== Experiment Complete! ===")
print(f"Results successfully saved to: {output_filename}")

display(df_results)

In [ ]:
import matplotlib.pyplot as plt

# 1. Calculate the independent variable (N * K)
df_results['Total_Processed_Bits'] = df_results['N_Bits'] * df_results['K_Iterations (Bags)']

# Sort the dataframe so the lines plot cleanly from left to right
df_sorted = df_results.sort_values(by='Total_Processed_Bits')

# 2. Set up the figure with 2 subplots (1 row, 2 columns)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Plot 1: Bandwidth vs Total Bits ---
ax1.plot(df_sorted['Total_Processed_Bits'], df_sorted['Bandwidth_KB'], 
         marker='o', linestyle='-', color='blue', label='Bandwidth (KB)')
ax1.set_title('Bandwidth vs. Total Bits (N * K)')
ax1.set_xlabel('Total Bits Processed (N * K)')
ax1.set_ylabel('Bandwidth (KB)')
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.legend()

# --- Plot 2: Time vs Total Bits ---
ax2.plot(df_sorted['Total_Processed_Bits'], df_sorted['Time_Seconds'], 
         marker='o', linestyle='-', color='red', label='Empirical Time (s)')
ax2.plot(df_sorted['Total_Processed_Bits'], df_sorted['theory_time(seconds)'], 
         marker='x', linestyle='--', color='green', label='Theoretical Time (s)')
ax2.set_title('Execution Time vs. Total Bits (N * K)')
ax2.set_xlabel('Total Bits Processed (N * K)')
ax2.set_ylabel('Time (Seconds)')
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()

# Display the plots
plt.tight_layout()
plt.show()

# Optional: Save the plot to a file
# fig.savefig('oprf_proportionality_check.png', dpi=300)